# PyMieDAP Updated Benchmark

This notebook mirrors the original benchmark flow with the current codebase.
It covers:

- aerosol Mie benchmarks
- atmospheric benchmark cases against the reference values in `pymiedap_benchmark.ipynb`
- Lambertian disk-integration benchmark and phase-curve plot

Launch it from the repo root with:

```bash
jupyter notebook examples/pymiedap_benchmark_updated.ipynb
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pymiedap").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pymiedap.pymiedap as pmd

NOTEBOOK_TOL = 5e-5
LAMBERT_TOL = 2e-4


def compare(name, actual, expected, tol=NOTEBOOK_TOL):
    actual = np.asarray(actual)
    expected = np.asarray(expected)
    diff = np.abs(actual - expected)
    return {
        "name": name,
        "max_abs_diff": float(np.max(diff)),
        "passed": bool(np.max(diff) <= tol),
    }


def show_results(results):
    for result in results:
        status = "PASS" if result["passed"] else "FAIL"
        print(f"{status:4} {result['name']:<20} max_abs_diff={result['max_abs_diff']:.8e}")


## Aerosol Mie Benchmark

In [ ]:
coordsx = [0, 1, 2, 3, 0, 2]
coordsy = [0, 1, 2, 3, 1, 3]

aerA = pmd.Aerosols(nr=[1.45], ni=[0], r_eff=0.23, v_eff=0.18)
aerB = pmd.Aerosols(nr=[1.44], ni=[0], r_eff=1.05, v_eff=0.07)
aerC = pmd.Aerosols(nr=[1.33], ni=[0], v_eff=0.07, r_eff=2, par3=0.5, psd='7')
aerD = pmd.Aerosols(nr=[1.33], ni=[0], r_eff=2.2, v_eff=0.07)

pmd.mie_code(aerA, [0.55], nsubr=40, ngaur=500)
pmd.mie_code(aerB, [0.55], nsubr=40, ngaur=500)
pmd.mie_code(aerC, [0.70], nsubr=40, ngaur=500)
pmd.mie_code(aerD, [0.70], nsubr=40, ngaur=500)

mie_results = [
    compare('aerA asym', aerA.asym, [0.72100229]),
    compare('aerB asym', aerB.asym, [0.7179972]),
    compare('aerC asym', aerC.asym, [0.804201]),
    compare('aerD asym', aerD.asym, [0.80188248]),
    compare('aerA coef l0', aerA.coefs[0, coordsx, coordsy, 0], [1.0, 0.0, 0.0, 0.92901809, 0.0, 0.0]),
    compare('aerA coef l10', aerA.coefs[0, coordsx, coordsy, 10], [0.08802913, 0.11121631, 0.10151652, 0.0832367, -0.00358954, 0.02614555]),
    compare('aerA coef l15', aerA.coefs[0, coordsx, coordsy, 15], [0.00228455, 0.00218155, 0.00224082, 0.00255047, 0.00033707, 0.00046371]),
    compare('aerA coef l24', aerA.coefs[0, coordsx, coordsy, 24], [5.48015333e-06, 6.77440344e-06, 2.37713944e-06, 1.97521301e-06, -7.37991466e-07, 3.53992457e-06]),
    compare('aerB coef l0', aerB.coefs[0, coordsx, coordsy, 0], [1.0, 0.0, 0.0, 0.86462372, 0.0, 0.0]),
    compare('aerB coef l10', aerB.coefs[0, coordsx, coordsy, 10], [5.06996479, 5.33408747, 5.31811435, 5.12294652, 0.03998552, 0.02898462]),
    compare('aerB coef l15', aerB.coefs[0, coordsx, coordsy, 15], [4.29325014, 4.22823484, 4.16063179, 4.35190093, -0.04425792, 0.31830627]),
    compare('aerB coef l24', aerB.coefs[0, coordsx, coordsy, 24], [1.48138437, 1.66704814, 1.50932983, 1.42450347, -0.04210442, 0.24849267]),
    compare('aerC coef l0', aerC.coefs[0, coordsx, coordsy, 0], [1.0, 0.0, 0.0, 0.95461472, 0.0, 0.0]),
    compare('aerC coef l10', aerC.coefs[0, coordsx, coordsy, 10], [1.17713563, 1.2847585, 1.2659546, 1.17411335, -0.02230815, 0.09498393]),
    compare('aerC coef l15', aerC.coefs[0, coordsx, coordsy, 15], [0.42561199, 0.42569832, 0.4231388, 0.4298012, -0.0055369, 0.04263078]),
    compare('aerC coef l80', aerC.coefs[0, coordsx, coordsy, 80], [1.28782521e-06, 1.40241107e-06, 1.04194570e-06, 9.98265353e-07, 8.49592384e-08, 4.62722533e-07]),
    compare('aerD coef l0', aerD.coefs[0, coordsx, coordsy, 0], [1.0, 0.0, 0.0, 0.91852089, 0.0, 0.0]),
    compare('aerD coef l10', aerD.coefs[0, coordsx, coordsy, 10], [7.20704159, 7.35295547, 7.25693617, 7.13468861, -0.029961, 0.09669141]),
    compare('aerD coef l15', aerD.coefs[0, coordsx, coordsy, 15], [7.98539793, 8.00874554, 7.98884822, 7.99970188, 0.02916147, 0.16475289]),
    compare('aerD coef l80', aerD.coefs[0, coordsx, coordsy, 80], [3.04667428e-03, 3.18324854e-03, 2.83399090e-03, 2.77620381e-03, 2.74463843e-05, 5.59438200e-04]),
]

show_results(mie_results)


## Atmospheric Benchmark Cases

In [ ]:
modelB = pmd.Model()
modelB.wvl_list = [0.7]
del modelB.layers.gasbelow
del modelB.layers.haze
modelB.layers.gastop.rayscat = False
modelB.layers.gastop.tau_ray = [0.1]
modelB.layers.cloud.rayscat = False
modelB.layers.cloud.tau = [0.4]
modelB.layers.cloud.tau_ray = [0.1]
modelB.layers.cloud.aerosols = aerC
modelB.dpol = 0.0279
modelB.surface[0, 0] = 0.1

modelA = pmd.Model()
modelA.wvl_list = [0.7]
del modelA.layers.gastop
del modelA.layers.haze
del modelA.layers.cloud
modelA.layers.gasbelow.rayscat = False
modelA.layers.gasbelow.tau = [1.0]
modelA.layers.gasbelow.aerosols = aerC
modelA.surface[0, 0] = 0.0

mus = np.array([0.1, 0.5, 1.0, 0.1, 0.5, 1.0, 0.1, 0.5, 1.0, 0.1, 0.5, 1.0])
emissions = np.degrees(np.arccos(mus))
mu0s = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
szas = np.degrees(np.arccos(mu0s))
dphi = np.radians([0, 0, 0, 30, 30, 30, 0, 0, 0, 30, 30, 30])
alphas = mus * mu0s + np.sqrt(1 - mus**2) * np.sqrt(1 - mu0s**2) * np.cos(dphi)
phases = np.degrees(np.arccos(alphas))
azimuth = pmd.calc_azimuth(phases, szas, emissions, deg=True)

geom_results = [
    compare('geometry phases', phases, [24.26082952, 8.53773646e-07, 60.0, 37.2274111, 25.9050793, 60.0, 0.0, 24.26082952, 84.26082952, 29.8461183, 37.2274111, 84.26082952]),
    compare('geometry emissions', emissions, [84.26082952, 60.0, 0.0, 84.26082952, 60.0, 0.0, 84.26082952, 60.0, 0.0, 84.26082952, 60.0, 0.0]),
    compare('geometry sza', szas, [60.0, 60.0, 60.0, 60.0, 60.0, 60.0, 84.26082952, 84.26082952, 84.26082952, 84.26082952, 84.26082952, 84.26082952]),
    compare('geometry azimuth', azimuth, [8.53773646e-07, 8.53773646e-07, 0.0, 30.0, 30.0, 0.0, 0.0, 8.53773646e-07, 0.0, 30.0, 30.0, 0.0]),
]
show_results(geom_results)

pmd.compute_model(modelA, output_name='benchA', rename=True, nmat=4, nmug=40)
pmd.compute_model(modelB, output_name='benchB', rename=True, nmat=4, nmug=40)

Ia, Qa, Ua, Va = pmd.read_dap_output(phases, szas, emissions, modelA.name[0], beta=np.zeros_like(dphi), phi=np.degrees(dphi))
Ib, Qb, Ub, Vb = pmd.read_dap_output(phases, szas, emissions, modelB.name[0], beta=np.zeros_like(dphi), phi=np.degrees(dphi))

atm_results = [
    compare('modelA bmsca', modelA.bmsca, np.zeros(50)),
    compare('modelA basca', modelA.basca, np.array([1.0] + [0.0] * 49)),
    compare('modelB bmsca', modelB.bmsca, np.array([0.1, 0.1] + [0.0] * 48)),
    compare('modelB basca', modelB.basca, np.array([0.4] + [0.0] * 49)),
    compare('modelA I', Ia * mu0s * np.pi, [1.10269, 0.31943, 0.033033, 0.66414, 0.25209, 0.033033, 2.93214, 0.22054, 0.009287, 0.76910, 0.132828, 0.009287]),
    compare('modelA Q', Qa * mu0s * np.pi, [0.004604, -0.002881, -0.002979, 0.000303, -0.001444, -0.001489, 0.009900, 0.000976, -0.000815, -0.003758, 0.000220, -0.000408]),
    compare('modelA U', Ua * mu0s * np.pi, [0.0, 0.0, 0.0, -0.002770, -0.004141, -0.002580, 0.0, 0.0, 0.0, 0.003124, -0.000525, -0.000706]),
    compare('modelA V', Va * mu0s * np.pi, [0.0, 0.0, 0.0, 0.000038, 0.000017, 0.0, 0.0, 0.0, 0.0, 0.000012, 0.000007, 0.0]),
    compare('modelB I', Ib * mu0s * np.pi, [0.53295, 0.20843, 0.093680, 0.41814, 0.18497, 0.093680, 0.52277, 0.106590, 0.026009, 0.27630, 0.083628, 0.026009]),
    compare('modelB Q', Qb * mu0s * np.pi, [-0.028340, -0.036299, -0.024156, -0.000058, -0.019649, -0.012078, 0.011506, -0.005186, -0.014984, 0.034368, 0.003839, -0.007492]),
    compare('modelB U', Ub * mu0s * np.pi, [0.0, 0.0, 0.0, -0.073105, -0.041401, -0.020920, 0.0, 0.0, 0.0, -0.016042, -0.014492, -0.012976]),
    compare('modelB V', Vb * mu0s * np.pi, [0.0, 0.0, 0.0, 0.000106, 0.000040, 0.0, 0.0, 0.0, 0.0, 0.000027, 0.000017, 0.0]),
]

show_results(atm_results)


## Lambertian Disk-Integration Benchmark

In [ ]:
modelL = pmd.Model()
modelL.wvl_list = [0.7]
del modelL.layers.gastop
del modelL.layers.haze
del modelL.layers.cloud
modelL.layers.gasbelow.rayscat = False
modelL.layers.gasbelow.tau = [0.0]
modelL.surface[0, 0] = 1.0

phase_radians = np.linspace(0.0, np.pi, 80)
phase_degrees = np.degrees(phase_radians)
theta = np.pi - phase_radians
lambert_phase = 2 * (np.sin(theta) - theta * np.cos(theta)) / (3.0 * np.pi)

pmd.planet_integrated([modelL], npix=60, alpha=phase_degrees, output_names=['benchLambert'])
lambert_residual = modelL.I[0, :] - lambert_phase

print(f'Max absolute Lambert residual: {np.max(np.abs(lambert_residual)):.8e}')
print('Non-finite indices:', np.where(~np.isfinite(lambert_residual))[0].tolist())
print('Pass:', np.max(np.abs(lambert_residual)) <= LAMBERT_TOL and np.isfinite(lambert_residual).all())


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

axes[0].plot(phase_degrees, modelL.I[0, :], label='PyMieDAP', linewidth=2)
axes[0].plot(phase_degrees, lambert_phase, label='Analytical Lambert', linestyle='--', linewidth=2)
axes[0].set_ylabel('Disk-Integrated I')
axes[0].set_title('Lambertian Phase Curve Benchmark')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(phase_degrees, lambert_residual, color='black', linewidth=1.5)
axes[1].axhline(0.0, color='tab:red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Phase Angle [deg]')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()

output_path = REPO_ROOT / 'examples' / 'lambert_phase_curve_notebook.png'
fig.savefig(output_path, dpi=180)
print(f'Saved figure to {output_path}')
plt.show()
